   

## Silver Layer

### `fact_orders`
**Historical Orders (Azure SQL DB via Lakeflow Connect) + Incremental Orders (Azure Event Hubs)**

**Grain:** One row per order

```sql
CREATE TABLE IF NOT EXISTS `02_silver`.fact_orders (
    order_id             STRING PRIMARY KEY,
    order_timestamp      TIMESTAMP,
    order_date           DATE,
    order_hour           INT,
    day_of_week          STRING,
    is_weekend           BOOLEAN,
    restaurant_id        STRING,
    customer_id          STRING,
    order_type           STRING,
    item_count           INT,
    total_amount         DECIMAL(10,2),
    payment_method       STRING,
    order_status         STRING,
    _ingestion_timestamp TIMESTAMP
);
```

   
### Step-1: Investigate 2 Tables - orders, historical_orders

In [0]:
historical_orders=spark.table("databricksproject_ws.`01_bronze`.historical_orders")
orders=spark.table("databricksproject_ws.`01_bronze`.orders")
display(historical_orders.limit(1))
display(orders.limit(1))

In [0]:
orders.printSchema()
historical_orders.printSchema()

   
## Observations

Observations after debugging above tables to construct fact_orders:
- Ensure `total_amount` is cast to decimal(10,2)
- Use `items_count` instead of `items`
- Add `ingestion_timestamp` for both DataFrames
- Extract time-related columns: `order_date`, `order_hour`, `day_of_week`, `is_weekend`

   
### Step-2: Combine 2 tables using `unionByName`

Use `unionByName` to merge DataFrames from two tables, aligning columns by name and filling missing columns with nulls if needed.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import DecimalType

# Transform streaming orders (items is already ARRAY<STRUCT>)
orders_df = (
    orders
        .withColumn("items_count", size(col("items")))
        .withColumn("total_amount", col("total_amount").cast(DecimalType(10, 2)))
        .withColumn("ingestion_timestamp", current_timestamp())
        .drop("items")
)

# Transform historical orders (items is STRING -> convert to ARRAY<STRUCT>)
ITEMS_SCHEMA = """                   
ARRAY<
    STRUCT<
        item_id: STRING,
        name: STRING,
        category: STRING,
        quantity: INT,
        unit_price: DECIMAL(10,2),
        subtotal: DECIMAL(10,2)
    >
>
"""

historical_orders_df = (
    historical_orders
        .withColumn("items", from_json(col("items"), ITEMS_SCHEMA))
        .withColumn("items_count", size(col("items")))
        .withColumn("total_amount", col("total_amount").cast(DecimalType(10, 2)))
        .withColumn("ingestion_timestamp", current_timestamp())
        .drop("items")
)

# Columns to retain
columns = [
    "order_id",
    "order_timestamp",
    "restaurant_id",
    "customer_id",
    "order_type",
    "items_count",
    "total_amount",
    "payment_method",
    "order_status",
    "ingestion_timestamp"
]

# Union historical + streaming data
union_df = (
    historical_orders_df
        .select(*columns)
        .unionByName(orders_df.select(*columns), allowMissingColumns=True)
)

display(union_df.limit(5))

In [0]:
from pyspark.sql.functions import to_timestamp, to_date, hour, date_format, when, col

union_df = (
    union_df
        .withColumn("order_timestamp", to_timestamp(col("order_timestamp")))
        .withColumn("order_date", to_date(col("order_timestamp")))
        .withColumn("order_hour", hour(col("order_timestamp")))
        .withColumn("day_of_week", date_format(col("order_timestamp"), "EEEE"))
        .withColumn("is_weekend", when(col("day_of_week").isin(["Saturday", "Sunday"]), True).otherwise(False))
)

display(union_df.select("order_id", "order_timestamp", "order_date", "order_hour", "day_of_week", "is_weekend", "restaurant_id", "customer_id", "order_type", "items_count", "total_amount", "payment_method", "order_status", "ingestion_timestamp").limit(5))

   
## SDP Final Code

   
### Silver Layer – `fact_orders` (AUTO CDC)

This pipeline creates the **`fact_orders`** Silver table by combining data from two Bronze streaming sources:

- **`historical_orders`** – Initial full load and ongoing changes from **Azure SQL Database** via **Lakeflow Connect**.
- **`orders`** – Real-time incremental order events from **Azure Event Hubs**.

Both sources are first **standardized to a common schema** by:
- Parsing the `items` JSON (historical source).
- Computing `items_count`.
- Casting `total_amount` to `DECIMAL(10,2)`.
- Deriving `order_date`, `order_hour`, `day_of_week`, and `is_weekend`.
- Adding `_ingestion_timestamp`.

The standardized streams are then combined into a **unified change stream** and processed using **Lakeflow AUTO CDC**.

**AUTO CDC Configuration**
- **Primary Key:** `order_id`
- **Sequence Column:** `last_updated`

AUTO CDC automatically:
- Inserts new orders.
- Updates existing orders when a newer `last_updated` arrives.
- Maintains a single, latest version of each order in the Silver table.
- Supports continuous incremental processing without creating duplicate records.
#### Flow

```text
                 Azure SQL Database                  Azure Event Hubs
                        │                                   │
                Lakeflow Connect                           │
                        │                                   │
                        ▼                                   ▼
        Bronze.historical_orders               Bronze.orders
                        │                                   │
                        ▼                                   ▼
              Standardize Schema                 Standardize Schema
                        └───────────────┬───────────────────┘
                                        │
                                        ▼
                               unified_orders
                                        │
                                        ▼
                  AUTO CDC (Key: order_id,
                            Sequence: last_updated)
                                        │
                                        ▼
                             Silver.fact_orders
```

In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import *
from pyspark.sql.types import DecimalType

ITEMS_SCHEMA = """
ARRAY<
    STRUCT<
        item_id: STRING,
        name: STRING,
        category: STRING,
        quantity: INT,
        unit_price: DECIMAL(10,2),
        subtotal: DECIMAL(10,2)
    >
>
"""

# ---------------------------------------
# Standardize Event Hub Orders
# ---------------------------------------
@dp.view
def orders_stream():

    return (
        spark.readStream.table("databricksproject_ws.`01_bronze`.orders")
            .withColumn("items_count", size(col("items")))
            .withColumn("total_amount", col("total_amount").cast(DecimalType(10,2)))
            .withColumn("order_timestamp", to_timestamp("order_timestamp"))
            .withColumn("order_date", to_date("order_timestamp"))
            .withColumn("order_hour", hour("order_timestamp"))
            .withColumn("day_of_week", date_format("order_timestamp", "EEEE"))
            .withColumn(
                "is_weekend",
                when(dayofweek("order_timestamp").isin(1, 7), True).otherwise(False)
            )
            .withColumn("_ingestion_timestamp", current_timestamp())
            .drop("items")
    )

# ---------------------------------------
# Standardize Historical Orders
# ---------------------------------------
@dp.view
def historical_orders_stream():

    return (
        spark.readStream.table("databricksproject_ws.`01_bronze`.historical_orders")
            .withColumn("items", from_json(col("items"), ITEMS_SCHEMA))
            .withColumn("items_count", size(col("items")))
            .withColumn("total_amount", col("total_amount").cast(DecimalType(10,2)))
            .withColumn("order_timestamp", to_timestamp("order_timestamp"))
            .withColumn("order_date", to_date("order_timestamp"))
            .withColumn("order_hour", hour("order_timestamp"))
            .withColumn("day_of_week", date_format("order_timestamp", "EEEE"))
            .withColumn(
                "is_weekend",
                when(dayofweek("order_timestamp").isin(1, 7), True).otherwise(False)
            )
            .withColumn("_ingestion_timestamp", current_timestamp())
            .drop("items")
    )

# ---------------------------------------
# Unified Change Stream
# ---------------------------------------
@dp.view
def unified_orders():

    return (
        dp.read_stream("historical_orders_stream")
            .unionByName(
                dp.read_stream("orders_stream"),
                allowMissingColumns=True
            )
    )

# ---------------------------------------
# Target Silver Table
# ---------------------------------------
dp.create_streaming_table(
    name="fact_orders"
)

# ---------------------------------------
# AUTO CDC
# ---------------------------------------
dp.create_auto_cdc_flow(
    target="fact_orders",
    source="unified_orders",
    keys=["order_id"],
    sequence_by=col("last_updated")
)